# Sammlung hilfreicher Funktionen für [Class](https://sites.wustl.edu/jeffheaton/t81-558/)

Dies ist eine Sammlung hilfreicher Funktionen, die ich im Laufe dieses Kurses vorstellen werde.

In [1]:
import base64
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from sklearn import preprocessing


# Kodieren Sie Textwerte in Dummyvariablen (z. B. [1,0,0], [0,1,0], [0,0,1] für Rot, Grün, Blau).
def encode_text_dummy(df, name):
    dummies = get_dummies(df[name])
    for x in dummies.columns:
        dummy_name = f"{name}-{x}"
        df[dummy_name] = dummies[x]
    df.drop(name, axis=1, inplace=True)


# Kodieren Sie Textwerte in eine einzelne Dummy-Variable. Die neuen Spalten (die die alten nicht ersetzen) haben eine 1
# an jeder Stelle, an der die ursprüngliche Spalte (Name) mit jedem der Zielwerte übereinstimmt. Eine Spalte wird hinzugefügt für
# für jeden Zielwert.
def encode_text_single_dummy(df, name, target_values):
    for tv in target_values:
        l = list(df[name].astype(str))
        l = [1 if str(x) == str(tv) else 0 for x in l]
        name2 = f"{name}-{tv}"
        df[name2] = l


# Kodieren Sie Textwerte in Indizes (z. B. [1], [2], [3] für Rot, Grün, Blau).
def encode_text_index(df, name):
    le = preprocessing.Labelcoder()
    df[name] = le.fit_transform(df[name])
    return le.classes_


# Eine numerische Spalte als Zscores kodieren
def encode_numeric_zscore(df, name, mean=None, sd=None):
    if mean is None:
        mean = df[name].mean()

    if sd is None:
        sd = df[name].std()

    df[name] = (df[name] - mean) / sd


# Konvertieren Sie alle fehlenden Werte in der angegebenen Spalte in den Median
def missing_median(df, name):
    med = df[name].median()
    df[name] = df[name].fillna(med)


# Konvertieren Sie alle fehlenden Werte in der angegebenen Spalte in den Standardwert
def missing_default(df, name, default_value):
    df[name] = df[name].fillna(default_value)


# Konvertieren Sie einen Pandas-Datenrahmen in die x,y-Eingaben, die TensorFlow benötigt
def to_xy(df, target):
    result = []
    for x in df.columns:
        if x != target:
            result.append(x)
    # finde den Typ der Zielspalte heraus. Ist das wirklich so schwer? :(
    target_type = df[target].dtypes
    target_type = target_type[0] if hasattr(target_type, "__iter__") else target_type
    # Zur Klassifizierung in int kodieren, andernfalls in float. TensorFlow bevorzugt 32 Bit.
    if target_type in (np.int64, np.int32):
        # Einstufung
        dummies = get_dummies(df[target])
        return df[result].values.astype(np.float32), dummies.values.astype(np.float32)
    # Regression
    return df[result].values.astype(np.float32), df[[target]].values.astype(np.float32)


# Schön formatierte Zeitzeichenfolge


def hms_string(sec_elapsed):
    h = int(sec_elapsed / (60 * 60))
    m = int((sec_elapsed % (60 * 60)) / 60)
    s = sec_elapsed % 60
    return f"{h}:{m:>02}:{s:>05.2f}"


# Regressionsdiagramm.
def chart_regression(pred, y, sort=True):
    t = DataFrame({"pred": pred, "y": y.flatten()})
    if sort:
        t.sort_values(by=["y"], inplace=True)
    plt.plot(t["y"].tolist(), label="expected")
    plt.plot(t["pred"].tolist(), label="prediction")
    plt.ylabel("output")
    plt.legend()
    plt.show()


# Entfernen Sie alle Zeilen, bei denen die angegebene Spalte +/- SD-Standardabweichungen beträgt


def remove_outliers(df, name, sd):
    drop_rows = df.index[(np.abs(df[name] - df[name].mean()) >= (sd * df[name].std()))]
    df.drop(drop_rows, axis=0, inplace=True)


# Kodieren Sie eine Spalte in einen Bereich zwischen normalized_low und normalized_high.
def encode_numeric_range(
    df, name, normalized_low=-1, normalized_high=1, data_low=None, data_high=None
):
    if data_low is None:
        data_low = min(df[name])
        data_high = max(df[name])

    df[name] = ((df[name] - data_low) / (data_high - data_low)) * (
        normalized_high - normalized_low
    ) + normalized_low